In [ ]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'


# import os
# os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
# os.environ["XLA_FLAGS"] = f"--xla_gpu_cuda_data_dir={os.environ.get('CUDA_HOME', '')}"

In [ ]:
from pathlib import Path
import pickle

import pandas as pd 
import matplotlib.pyplot as plt

In [ ]:
run_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/runs/sf_grnn_pooled/era5_vae_20260521_231506")

results = pd.read_parquet(run_dir / 'results' / 'test_results.parquet')
with open(run_dir / 'results' / "test_metrics.pkl", 'rb') as f:
    bulk_m, basin_m = pickle.load(f)

# bulk_m

In [ ]:
from evaluate.plots import mosaic_scatter

fig = mosaic_scatter(results, bulk_m)

In [ ]:
from evaluate.plots import basin_metric_histograms

basin_metric_histograms(basin_m)

In [ ]:
basin_m['discharge'].sort_values('KGE').dropna()

In [ ]:
basin = 'USGS-14211820'

df = results.xs(basin, level='subbasin').sort_index()
df = df.droplevel(0, axis=0)
df.columns = ['_'.join(ml_col) for ml_col in df.columns]
df = df[df.index > pd.Timestamp('2020-12-31')]


fig, ax = plt.subplots(1,1, figsize=(6,4))
df['obs_discharge'].plot(ax=ax, label='Gauge')
df['pred_discharge'].plot(ax=ax, label='Model')
# if 'pred_discharge_std' in df.columns:
#     x = df.index
#     y = df['pred_discharge']
#     error = df['pred_discharge_std']
    # plt.fill_between(x, low, high, alpha=0.4, facecolor='tab:orange', antialiased=True)

plt.legend()
ax.set_ylabel('Discharge [m³/s]')
ax.set_xlabel('Date')
plt.show()
# fig.savefig(run_dir / 'figures' / f"timeseries_{basin}.png", dpi=300)

In [ ]:
list(pd.read_parquet('/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual/attributes/static.parquet'))

In [ ]:
bad_gauges = pd.read_csv('/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual/basin_lists/bad_gauges.txt')['site_id'].to_list()
bad_gauges
basin_filt = basin_m[~basin_m.index.isin(bad_gauges)]
kge_sort = basin_filt['discharge'].sort_values('KGE').dropna()
kge_sort

In [ ]:
for site, row in kge_sort.iloc[:20].iterrows():
    df = results.xs(site, level='subbasin').sort_index()
    df = df.droplevel(0, axis=0)
    df.columns = ['_'.join(ml_col) for ml_col in df.columns]
    df = df[df.index > pd.Timestamp('2020-12-31')]
    
    fig, ax = plt.subplots(1,1, figsize=(6,4))
    df['obs_discharge'].plot(ax=ax, label='Gauge')
    df['pred_discharge'].plot(ax=ax, label='Model')
    
    plt.legend()
    try:
        basin_name = sites.loc[basin]['name']
    except:
        basin_name = 'N/A'
    ax.set_title(f"{basin}: {basin_name}")
    ax.set_ylabel('Discharge [m³/s]')
    ax.set_xlabel('Date')
    print(site)
    print(row['KGE'])
    plt.show()

In [ ]:
results['obs']['discharge'].dropna()

In [ ]:
plt.scatter(df['obs']['discharge'], df['pred']['discharge'])

In [ ]:
run_dir

In [ ]:
targets = list(bulk_m.keys())

In [ ]:
import geopandas as gpd
proj_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/distributed")
metadata_dir = proj_dir / "metadata"
basins = gpd.read_file(metadata_dir / 'matchups.geojson').set_index("HYBAS_ID")
basins.index = basins.index.astype(str)
basins

In [ ]:
import evaluate
pred = evaluate.predict(trainer.model, dataloader, denormalize=False)

In [ ]:
pred

In [ ]:
x = pred.xs('2021-02-08', level='date')


x['pred']['discharge']

In [ ]:
x['pred']['discharge'].hist()

In [ ]:
basins_pred = basins.merge(x['pred']['discharge'], left_index=True, right_index=True)
basins_pred.plot('discharge')

In [ ]:
plt.scatter(x['obs']['discharge'], x['pred']['discharge'])
# plt.plot([0,500], [0,500], 'r--')